In [16]:
### RAG pipeline -Data ingestion to vector DB pipeline

import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path 


 

In [18]:
### Read all the pdfs inside the directory

def process_all_pdfs(pdf_directory):
    """Process all pdf files ina a directory"""
    all_documents=[]
    pdf_dir=Path(pdf_directory) 
    
    #find all the pdf files recursively 
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    
    print (f"found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\n Processing :{pdf_file.name }")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()
            
            
            #add source information to metadata 
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
                
            
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")
        
        except Exception as e:
            print (f" Error {e}")
            
    
    print (f"\n Total documents loaded :{len(all_documents)}")
    return all_documents 
            
            
#process all pdfs in the data directory
all_pdf_documents=process_all_pdfs("../data")
                
    
    
     

found 1 PDF files to process

 Processing :AdityaResume.pdf
 Loaded 1 pages

 Total documents loaded :1


In [19]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-03T18:33:12+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-03T18:33:12+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/AdityaResume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'AdityaResume.pdf', 'file_type': 'pdf'}, page_content='Aditya Prasannan\n+91-8955833540 | adityaprasannan.1si21cs004@gmail.com | LinkedIn | Github | Leetcode\nEducation\n• Siddaganga Institute Of Technology 2021-2025\nBachelor of Engineering in Computer Science CGPA: 8.98\n• Cambridge Court High School 2016-2020\nJaipur,Rajasthan\nExperience\n• Oracle June 2025-Present\nAssociate Software Engineer Bengaluru\n– Designed and implemented an AI-driven solution to automatically generate standardized Business Requirement\nDocuments (BRDs)

In [21]:
##text splitting get into chunks 

def split_documents(documents,chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance """
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")
    
    
    #example of a chunk 
    
    if split_docs:
        print(f"\n example chunk")
        print(f"Content : {split_docs[0].page_content[:200]}..")
        print(f"metadata: {split_docs[0].metadata}")
    
    return split_docs
    

In [22]:
chunks =split_documents(all_pdf_documents)

split 1 documents into 4 chunks

 example chunk
Content : Aditya Prasannan
+91-8955833540 | adityaprasannan.1si21cs004@gmail.com | LinkedIn | Github | Leetcode
Education
• Siddaganga Institute Of Technology 2021-2025
Bachelor of Engineering in Computer Scien..
metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-03T18:33:12+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-03T18:33:12+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/AdityaResume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'AdityaResume.pdf', 'file_type': 'pdf'}


In [24]:
###embedding and vector storeDB

import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity 


In [ ]:
class EmbeddingManager:
    """handles document embedding generation using Sentence TRansformer """
    
    def __init__(self,model_name: str= "all-MiniLM-L6-v2"):
        """Iniitialize the embedding manager 
        args: model_name:huggingFace model name for sentence embedding 
        """
        self.model_name=model_name
        self.model=None
        self._load_model()
        
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model :{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully .Embedding dimension :{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise 
        
    def generate_embeddings(self,texts:List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts 
        args: texts: List of text strings to embed
        Returns: numpy array of embeddings with shape (len(texts),embedding_dim)
         
        """
        
        if not self.model:
            raise ValueError("Model not loaded ")
        
        print(f"Generating embeddings for {len(texts)} texts ...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape :{embeddings.shape}")
        return embeddings 
            
    # def get_embedding_dimension(self) ->int:
    #     """get the embedding dimension of the model  """
    #     if not self.model:
    #         raise ValueError("model not loaded")
    #     return self.model.get_sentence_embedding_dimension()
    
##initialize embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

        
         

Loading embedding model :all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8050.04it/s]


Model loaded successfully .Embedding dimension :384


/var/folders/g1/78gjyx710bzgzhjb28gs9_k80000gn/T/ipykernel_17685/387117672.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully .Embedding dimension :{self.model.get_sentence_embedding_dimension()}")
